
# BigQuery External Tables

This notebook creates **external tables** in **BigQuery** pointing to CSV files in **Google Cloud Storage (GCS)** and runs sample queries. It also demonstrates that **DML (INSERT/UPDATE)** operations are **not supported** on external tables by intentionally attempting them and observing the errors.

> **Note:** External tables are *read-only*. You can query them, but you **cannot** modify their data using DML.



## Prerequisites

- You have a Google Cloud project (e.g., `pp-bigquery-02`) with **BigQuery** and **GCS** access.
- The CSV files are accessible to your project:
  - `gs://spark-training-pp/retail_db/orders/orders.csv`
  - `gs://spark-training-pp/retail_db/order_items/order_items.csv`
- You are running this notebook in an environment where the `%%bigquery` magic is available (e.g., Colab or Jupyter with the BigQuery extension).


## Create External Table: `order_items`

In [ ]:

%%bigquery
-- Create or replace external table for order_items
CREATE OR REPLACE EXTERNAL TABLE `pp-bigquery-02.demo_external_tables.order_items`
(
  order_item_id INT64,
  order_item_order_id INT64,
  order_item_product_id INT64,
  order_item_quantity INT64,
  order_item_subtotal FLOAT64,
  order_item_product_price FLOAT64
)
OPTIONS (
  format = 'CSV',
  uris = ['gs://spark-training-pp/retail_db/order_items/order_items.csv'],
  skip_leading_rows = 1,
  field_delimiter = ','
);



## Create External Table: `orders`


In [ ]:

%%bigquery
-- Create or replace external table for orders
CREATE OR REPLACE EXTERNAL TABLE `pp-bigquery-02.demo_external_tables.orders`
(
  order_id INT64,
  order_date TIMESTAMP,
  order_customer_id INT64,
  order_status STRING
)
OPTIONS (
  format = 'CSV',
  uris = ['gs://spark-training-pp/retail_db/orders/orders.csv'],
  skip_leading_rows = 1,
  field_delimiter = ','
);



## Simple Projection


In [ ]:

%%bigquery
SELECT
    order_item_id AS id,
    order_item_order_id AS oid
FROM
    `pp-bigquery-02.demo_external_tables.order_items`
LIMIT 10;



## Revenue by Date & Product


In [ ]:

%%bigquery
SELECT
  o.order_date,
  oi.order_item_product_id,
  ROUND(SUM(oi.order_item_subtotal), 2) AS revenue
FROM
  `pp-bigquery-02.demo_external_tables.orders` AS o
JOIN
  `pp-bigquery-02.demo_external_tables.order_items` AS oi
ON
  o.order_id = oi.order_item_order_id
WHERE
  o.order_status IN ('COMPLETE', 'CLOSED')
GROUP BY
  1, 2
ORDER BY
  1, 3 DESC;



## External Tables Are Read‑Only

External tables in BigQuery **do not support DML** operations like `INSERT`, `UPDATE`, `DELETE`, or `MERGE`.  
The following cells intentionally attempt DML and will **fail** with an error (expected).

In [ ]:

%%bigquery
INSERT INTO `pp-bigquery-02.demo_external_tables.order_items` (order_item_id, order_item_order_id, order_item_product_id, order_item_quantity, order_item_subtotal, order_item_product_price)
VALUES (99999, 99999, 42, 1, 19.99, 19.99);


In [ ]:

%%bigquery
UPDATE `pp-bigquery-02.demo_external_tables.orders`
SET order_status = 'CLOSED'
WHERE order_id = 1;



## Tips

- If you need to **modify** data, load the external data into a **native BigQuery table** 
- Keep external tables for **cost‑effective reads** over large files in GCS or for **schema‑on‑read** scenarios.
